## Running multiple circuits at once with `CircuitBinding`

This notebooks aims at presenting `CircuitBinding`. This feature lets you
multiple combinations of circuits, parameter values and measurement schemes in a
single job and being able to retrieved the correct result with the run. This has
the desired effect of reducing costs of sending circuits to providers and in
certain cases, even get faster runs.

This feature is the MPQP equivalent to qiskit's `pubs` and braket's `ProgramSets`.

The term binding comes from the fact that you can "bind" a circuit with
measurements (for example observables) and variables (in case of parametrized
circuits) so that you can run for instance one the circuit with different observables.

In [1]:
# First we setup here the circuits, values and observables to be used
from mpqp import BindingMode, CircuitBinding
from mpqp.execution.devices import (
    IBMDevice,
    AWSDevice,
)

from mpqp import (
    CNOT,
    BasisMeasure,
    ExpectationMeasure,
    H,
    IBMDevice,
    Observable,
    QCircuit,
    U,
    pI,
    pZ,
    pX,
    run,
)
from sympy import Symbol
from mpqp.execution.runner import run
from mpqp.execution.devices import IBMDevice, AWSDevice
import numpy as np

theta, phi, psi = Symbol('θ'), Symbol('phi'), Symbol('psi')
c1 = QCircuit([U(theta, phi, psi, 0)], label="c1")
c2 = QCircuit([H(0)], label="c2")
c3 = QCircuit([H(0), CNOT(0, 1)], label="c3")
c4 = QCircuit([H(0), H(1), CNOT(0, 1)], label="c4")

v1 = {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
v2 = {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
v3 = {'θ': 3.0, 'phi': 3.0, 'psi': 3.0}
v4 = {'θ': 4.0, 'phi': 4.0, 'psi': 4.0}
v_t0 = {'θ': 0}
v_t1 = {'θ': np.pi}

m1 = ExpectationMeasure(Observable(pI), label="Exp1", shots=2024)
m1_bis = ExpectationMeasure(Observable(pZ), label="Exp1bis", shots=2024)
m2 = ExpectationMeasure(Observable(pX @ pZ), label="Exp2", shots=2024)
m2_bis = ExpectationMeasure(Observable(pZ @ pZ), label="Exp2bis", shots=2024)
m3 = BasisMeasure(label="b3", shots=2024)
m3_bis = BasisMeasure(label="b3_bis", shots=1024)
m4 = None

m_I = ExpectationMeasure(Observable(pI), label="Exp_I", shots=2024)
m_Z = ExpectationMeasure(Observable(pZ), label="Exp_Z", shots=2024)

### Usecase 1: Running a list of circuit

The most basic case is the one where we just want to run circuits without any binding. In this case any `BasisMeasure` or `ExpectationMeasure` are embedded in the circuits and not provided in the `CircuitBinding` object.

In [2]:
# Example 1: Two state vector jobs
# Note: state vector jobs are not supported by braket's ProgramSets
binding = CircuitBinding([c3, c4])
print(run(binding, IBMDevice.AER_SIMULATOR))

BatchResult: 2 results
    Result: c3, IBMDevice, AER_SIMULATOR
      State vector: [0.70711, 0, 0, 0.70711]
      Probabilities: [0.5, 0, 0, 0.5]
      Number of qubits: 2
    Result: c4, IBMDevice, AER_SIMULATOR
      State vector: [0.5, 0.5, 0.5, 0.5]
      Probabilities: [0.25, 0.25, 0.25, 0.25]
      Number of qubits: 2


In [3]:
binding_with_basismeasure = CircuitBinding([c3 + QCircuit([m3]), c4 + QCircuit([m3])])
# or you can do
binding_with_basismeasure2 = CircuitBinding(
    [c3, c4], measurements=[m3], mode=BindingMode.PRODUCT
)
print(run(binding_with_basismeasure, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 2 results
    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Counts: [1010, 0, 0, 1014]
      Probabilities: [0.49901, 0, 0, 0.50099]
      Samples:
        State: 00, Index: 0, Count: 1010, Probability: 0.4990119
        State: 11, Index: 3, Count: 1014, Probability: 0.5009881
      Error: None
    Result: c4, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Counts: [515, 527, 462, 520]
      Probabilities: [0.25445, 0.26038, 0.22826, 0.25692]
      Samples:
        State: 00, Index: 0, Count: 515, Probability: 0.2544466
        State: 01, Index: 1, Count: 527, Probability: 0.2603755
        State: 10, Index: 2, Count: 462, Probability: 0.2282609
        State: 11, Index: 3, Count: 520, Probability: 0.256917
      Error: None


In [4]:
binding_with_observables = CircuitBinding([c3 + QCircuit([m2]), c4 + QCircuit([m2])])

print(run(binding_with_observables, IBMDevice.AER_SIMULATOR))
print(run(binding_with_observables, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 2 results
    Result: c3, IBMDevice, AER_SIMULATOR
      Expectation value: 0.023209876543209877
      Error/Variance: 0.0222162358784702
      Observables: [pX@pZ]
    Result: c4, IBMDevice, AER_SIMULATOR
      Expectation value: -0.009382716049382716
      Error/Variance: 0.02222124403002106
      Observables: [pX@pZ]
BatchResult: 2 results
    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Expectation value: 0.003952569169960474
      Error/Variance: 0.0004940634274688521
      Observables: [pX@pZ]
    Result: c4, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Expectation value: 0.016798418972332016
      Error/Variance: 0.0004939317258498172
      Observables: [pX@pZ]


### Usecase 2: Binding modes
#### BindingMode.PRODUCT:
For the first mode there is the product which is simply the products between the circuits, measurements and parameters.  
If we have `CircuitBinding([c1,c2], [m1,m2], [p1,p2], BindingMode.PRODUCT)` it results in the following executables:  
c1-p1-m1, c1-p1-m2, c1-p2-m1, c1-p2-m2, c2-p1-m1, c2-p1-m2, c2-p2-m1, c2-p2-m2.

In [5]:
binding_product = CircuitBinding(
    [c1, c2], values=[v1], measurements=[m1, m1_bis], mode=BindingMode.PRODUCT
)

print(run(binding_product, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 4 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 0.532608695652174
      Error/Variance: 0.0003539169848397677
      Observables: [pZ]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: -0.0266798418972332
      Error/Variance: 0.0004937194595041199
      Observables: [pZ]


#### BindingMode.ZIP
This binding mode has more subtleties than the product mode, it is split between 2 cases: if the number of Circuits (or CircuitBinding) is either 1 or n which is zipped with a n number of parameters and measurements. It's akin to the zip method between two lists.
If we have `CircuitBinding([c1,c2], [p1,p2], [m1,m2], BindingMode.ZIP)` it results in the following executables: c1-p1-m1, c2-p2-m2.
If we have 1 circuit then it is as follow: `CircuitBinding([c1], [p1,p2], [m1,m2], BindingMode.ZIP)` we have: c1-p1-m1, c1-p2-m2.

Note: if the length of the list circuits is different of the length of the lists measurements and values then it will raise an Error, we can only have the lengths (1,n,n) or (n,n,n)

In [6]:
binding_zip = CircuitBinding(
    [c1, c2], values=[v1, v2], measurements=[m1, m1_bis], mode=BindingMode.ZIP
)

print(run(binding_zip, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 2 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.023715415019762844
      Error/Variance: 0.000493793270301601
      Observables: [pZ]


In [7]:
binding_zip_1_circuit = CircuitBinding(
    [CircuitBinding([c1, c2])],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.ZIP,
)

print(run(binding_zip_1_circuit, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 4 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.39822134387351776
      Error/Variance: 0.00041572122593061734
      Observables: [pZ]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.001976284584980237
      Error/Variance: 0.0004940692165510075
      Observables: [pZ]


Note: There is an edge case if the circuit list only contains 1 CircuitBinding then it'll act like it's one circuit but iterated through every circuits in the embedded CircuitBinding.

In [8]:
binding_zip_1_cb = CircuitBinding([c1], [v1, v2], [m1, m1_bis], mode=BindingMode.ZIP)

print(run(binding_zip_1_cb, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 2 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.41106719367588934
      Error/Variance: 0.00041058486278825536
      Observables: [pZ]


### Usecase 3: Embedded CircuitBindings.

In the case of more complex desired binding it is possible to have a CircuitBinding inside another CircuitBinding (in the circuits list). In this case the checks in the ZIP mode consider a circuitBinding as 1 circuit.  
In the case we bind a measurement to an embedded circuitbinding it outputs:  
`CircuitBinding([CircuitBinding([c1,c2], [p1,p2], PRODUCT), c3], [m1,m2], ZIP)` => c1-p1-m1, c1-p2-m1, c2-p1-m1, c2-p2-m1, c3-m2.

As we see if one of the parameter is not inputted in the embedded circuitbinding it merges the outer parameters with the inner ones. However if both are reference it will proceed like this:  
- inner parameters-measurement bind  
- inner parameters-outer measurements  
- outer parameters-inner measurements  
- outer parameters-outer measurements  

Example: `CircuitBinding([CircuitBinding(c1, p1, m1), c2], [p2], m2, PRODUCT)`  
c1-p1-m1, c1-p2-m1, c1-p1-m2, c1-p2-m2, c2-p2-m2

In [9]:
binding_embedded = CircuitBinding(
    [CircuitBinding([c1], values=[v1], measurements=m1), c2],
    values=[v2],
    measurements=[m1_bis],
    mode=BindingMode.PRODUCT,
)

print(run(binding_embedded, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 5 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.38735177865612647
      Error/Variance: 0.00041994001955135136
      Observables: [pZ]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 0.5217391304347826
      Error/Variance: 0.00035957918961124354
      Observables: [pZ]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: 0.021739130434782608
      Error/Variance: 0.00049383765